# Cadence - Evaluation Pipeline

## 1. Install dependencies

In [32]:
%pip install --upgrade ai-edge-litert librosa soundfile resampy pandas matplotlib -q

In [33]:
from dataclasses import dataclass
from pathlib import Path
import math
import shutil
import time
from typing import Any

from ai_edge_litert.interpreter import Interpreter
import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

SAMPLE_RATE = 16_000
CHUNK_DURATION_S = 10.0
CHUNK_SIZE = int(SAMPLE_RATE * CHUNK_DURATION_S)

# See: src/ml/indicatorExtractor.ts
FRAME_SIZE = 512
ADAPTIVE_THRESHOLD_MULTIPLIER = 2.5
MIN_SILENCE_THRESHOLD = 0.001
MIN_PAUSE_FRAMES = 8

MIN_AGGREGATE_CHUNK_DURATION_S = 8.0
MANIFEST_PATH = Path("/content/cadence_dataset_samples/metadata/dataset_manifest.csv")
MODEL_PATH = Path("/content/cadence_outputs/encoder_wav2vec2_int8.tflite")
OUTPUT_DIR = Path("/content/cadence_eval_outputs")
LOCAL_EVAL_WAV_DIR = Path("/content/cadence_eval_work/wav")

LOCAL_EVAL_WAV_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 80)
print("Evaluation runtime ready")
print(f"Manifest: {MANIFEST_PATH}")
print(f"Model: {MODEL_PATH}")
print(f"Output: {OUTPUT_DIR}")

Evaluation runtime ready
Manifest: /content/cadence_dataset_samples/metadata/dataset_manifest.csv
Model: /content/cadence_outputs/encoder_wav2vec2_int8.tflite
Output: /content/cadence_eval_outputs


## 2. Load audio and normalize

In [34]:
def load_audio(path: str | Path) -> tuple[np.ndarray, float]:
    # .load decodes, resamples, and downmixes to mono.
    # NOTE: If the input is stereo, librosa averages channels into one mono vector before returning it
    pcm, sr = librosa.load(path, sr=SAMPLE_RATE, mono=True, dtype=np.float32)

    if sr != SAMPLE_RATE:
        raise ValueError(f"Expected {SAMPLE_RATE} Hz after resampling, got {sr} Hz")
    if pcm.ndim != 1:
        raise ValueError(f"Expected mono PCM vector, got shape {pcm.shape}")

    peak = float(np.max(np.abs(pcm))) if pcm.size else 0.0
    print(
        f"Loaded {path}: samples={len(pcm)}, duration={len(pcm) / SAMPLE_RATE:.3f}s, peak={peak:.6f}"
    )
    if peak > 1.0 + 1e-6:
        raise ValueError(f"PCM peak {peak:.6f} exceeds expected [-1, 1] range")

    # The returned PCM should already be float32 in [-1.0, 1.0], matching the app's decoded PCM contract.
    return pcm.astype(np.float32, copy=False), len(pcm) / SAMPLE_RATE

## 3. Chunking with timestamp tracking

In [35]:
@dataclass
class ChunkRecord:
    chunk_index: int
    start_time_s: float
    end_time_s: float
    true_duration_s: float
    is_padded: bool
    pcm: np.ndarray


def chunk_audio(pcm: np.ndarray, total_duration_s: float) -> list[ChunkRecord]:
    if pcm.ndim != 1:
        raise ValueError(f"Expected mono PCM vector, got shape {pcm.shape}")
    if len(pcm) == 0:
        return []

    chunk_count = math.ceil(len(pcm) / CHUNK_SIZE)
    chunks: list[ChunkRecord] = []

    for chunk_index in range(chunk_count):
        sample_start = chunk_index * CHUNK_SIZE
        sample_end = min(sample_start + CHUNK_SIZE, len(pcm))
        true_samples = sample_end - sample_start
        padded = np.zeros(CHUNK_SIZE, dtype=np.float32)
        padded[:true_samples] = pcm[sample_start:sample_end]

        start_time_s = chunk_index * CHUNK_DURATION_S
        end_time_s = min((chunk_index + 1) * CHUNK_DURATION_S, total_duration_s)
        true_duration_s = max(0.0, end_time_s - start_time_s)

        chunks.append(
            ChunkRecord(
                chunk_index=chunk_index,
                start_time_s=start_time_s,
                end_time_s=end_time_s,
                true_duration_s=true_duration_s,
                is_padded=true_samples < CHUNK_SIZE,
                pcm=padded,
            )
        )

    print(
        f"Chunked audio: {len(chunks)} chunks, final true duration={chunks[-1].true_duration_s:.3f}s"
    )
    return chunks

## 4. LiteRT inference per chunk

In [36]:
def load_interpreter(model_path: str | Path) -> Interpreter:
    interpreter = Interpreter(model_path=str(model_path))
    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]

    print(
        "Input:",
        input_details["shape"],
        input_details["dtype"],
        "quant=",
        input_details.get("quantization"),
    )

    print(
        "Output:",
        output_details["shape"],
        output_details["dtype"],
        "quant=",
        output_details.get("quantization"),
    )

    return interpreter


def _prepare_input_tensor(
    chunk_array: np.ndarray, input_details: dict[str, Any]
) -> np.ndarray:
    chunk = chunk_array.astype(np.float32, copy=False).reshape(1, CHUNK_SIZE)
    dtype = input_details["dtype"]

    if dtype == np.float32:
        return chunk

    scale, zero_point = input_details.get("quantization", (0.0, 0))
    if scale and scale > 0:
        quantized = np.round(chunk / scale + zero_point)
        info = np.iinfo(dtype)
        return np.clip(quantized, info.min, info.max).astype(dtype)

    return chunk.astype(dtype)


def run_inference(
    interpreter: Interpreter, chunk_array: np.ndarray
) -> tuple[np.ndarray, float]:
    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]
    input_tensor = _prepare_input_tensor(chunk_array, input_details)

    start = time.perf_counter()
    interpreter.set_tensor(input_details["index"], input_tensor)
    interpreter.invoke()
    output = interpreter.get_tensor(output_details["index"])
    elapsed_s = time.perf_counter() - start

    embeddings = np.asarray(output).squeeze(axis=0).astype(np.float32, copy=False)
    if embeddings.shape != (500, 768):
        raise ValueError(
            f"Expected embeddings shape (500, 768), got {embeddings.shape}"
        )

    return embeddings, elapsed_s

## 5. Indicator extraction

In [ ]:
def _frame_rms_values(pcm: np.ndarray, true_duration_s: float) -> np.ndarray:
    real_sample_count = min(len(pcm), max(0, round(true_duration_s * SAMPLE_RATE)))
    frame_count = (
        math.ceil(real_sample_count / FRAME_SIZE) if real_sample_count > 0 else 0
    )
    values = []

    for frame_index in range(frame_count):
        frame_start = frame_index * FRAME_SIZE
        frame_end = min(frame_start + FRAME_SIZE, real_sample_count)
        frame = pcm[frame_start:frame_end]
        values.append(float(np.sqrt(np.mean(np.square(frame)))) if frame.size else 0.0)

    return np.asarray(values, dtype=np.float64)


def _estimate_noise_floor(frame_rms: np.ndarray) -> float:
    if frame_rms.size == 0:
        return 0.0
    sorted_values = np.sort(frame_rms)
    index = math.floor(len(frame_rms) * 0.1)
    return float(sorted_values[min(index, len(sorted_values) - 1)])


def extract_pcm_indicators(
    pcm: np.ndarray, true_duration_s: float = CHUNK_DURATION_S
) -> dict[str, float | int]:
    frame_rms = _frame_rms_values(pcm, true_duration_s)
    frame_count = int(frame_rms.size)
    noise_floor = _estimate_noise_floor(frame_rms)
    silence_threshold = max(
        MIN_SILENCE_THRESHOLD, noise_floor * ADAPTIVE_THRESHOLD_MULTIPLIER
    )

    speech_frame_count = 0
    silence_frame_count = 0
    pause_count = 0
    pause_frames = 0
    longest_pause_frames = 0
    current_silence_run = 0

    for frame_value in frame_rms:
        if frame_value < silence_threshold:
            silence_frame_count += 1
            current_silence_run += 1
            continue

        speech_frame_count += 1
        if current_silence_run >= MIN_PAUSE_FRAMES:
            pause_count += 1
            pause_frames += current_silence_run
            longest_pause_frames = max(longest_pause_frames, current_silence_run)
        current_silence_run = 0

    if current_silence_run >= MIN_PAUSE_FRAMES:
        pause_count += 1
        pause_frames += current_silence_run
        longest_pause_frames = max(longest_pause_frames, current_silence_run)

    mean_rms = float(np.mean(frame_rms)) if frame_count else 0.0
    std_rms = float(np.std(frame_rms)) if frame_count else 0.0
    speech_activity_ratio = speech_frame_count / frame_count if frame_count else 0.0

    return {
        "frame_count": frame_count,
        "mean_rms": mean_rms,
        "std_rms": std_rms,
        "min_rms": float(np.min(frame_rms)) if frame_count else 0.0,
        "max_rms": float(np.max(frame_rms)) if frame_count else 0.0,
        "noise_floor": noise_floor,
        "silence_threshold": silence_threshold,
        "pause_count": pause_count,
        "total_pause_duration_s": (pause_frames * FRAME_SIZE) / SAMPLE_RATE,
        "longest_pause_s": (longest_pause_frames * FRAME_SIZE) / SAMPLE_RATE,
        "speech_frame_count": speech_frame_count,
        "silence_frame_count": silence_frame_count,
        "speech_activity_ratio": speech_activity_ratio,
    }


def extract_embedding_indicators(embeddings: np.ndarray) -> dict[str, float]:
    if embeddings.shape != (500, 768):
        raise ValueError(
            f"Expected embeddings shape (500, 768), got {embeddings.shape}"
        )

    # Prosody proxy only: this is not true pitch extraction. Mirrors the app's
    # embeddingStd field: mean per-dimension standard deviation over time.
    per_dimension_std = np.std(embeddings, axis=0)
    return {"embedding_std": float(np.mean(per_dimension_std))}

## 6. Run evaluation loop w/ aggregation & results table

In [ ]:
def evaluate_audio_file(
    audio_path: str | Path,
    model_path: str | Path,
    interpreter: Interpreter | None = None,
) -> tuple[pd.DataFrame, float]:
    pcm, total_duration_s = load_audio(audio_path)
    chunks = chunk_audio(pcm, total_duration_s)
    active_interpreter = (
        interpreter if interpreter is not None else load_interpreter(model_path)
    )
    records: list[dict[str, Any]] = []

    for chunk in chunks:
        embeddings, inference_time_s = run_inference(active_interpreter, chunk.pcm)
        pcm_indicators = extract_pcm_indicators(chunk.pcm, chunk.true_duration_s)
        embedding_indicators = extract_embedding_indicators(embeddings)

        records.append(
            {
                "chunk_index": chunk.chunk_index,
                "start_time_s": chunk.start_time_s,
                "end_time_s": chunk.end_time_s,
                "true_duration_s": chunk.true_duration_s,
                "is_padded": chunk.is_padded,
                "aggregate_included": chunk.true_duration_s
                >= MIN_AGGREGATE_CHUNK_DURATION_S,
                "inference_time_s": inference_time_s,
                **pcm_indicators,
                **embedding_indicators,
            }
        )
        print(
            f"Chunk {chunk.chunk_index + 1}/{len(chunks)}: "
            f"inference={inference_time_s:.3f}s, "
            f"speech={records[-1]['speech_activity_ratio']:.3f}, "
            f"mean_rms={records[-1]['mean_rms']:.6f}"
        )

    return pd.DataFrame.from_records(records), total_duration_s


def session_summary(
    df: pd.DataFrame, total_duration_s: float | None = None
) -> pd.DataFrame:
    included = df[df["aggregate_included"]].copy()
    if included.empty:
        included = df.copy()

    total_pause_count = int(included["pause_count"].sum()) if not included.empty else 0
    total_pause_duration_s = (
        float(included["total_pause_duration_s"].sum()) if not included.empty else 0.0
    )
    mean_pause_duration_s = (
        total_pause_duration_s / total_pause_count if total_pause_count else 0.0
    )

    summary = {
        "total_session_duration_s": float(
            total_duration_s if total_duration_s is not None else df["end_time_s"].max()
        ),
        "chunk_count": int(len(df)),
        "aggregate_chunk_count": int(len(included)),
        "total_inference_time_s": float(df["inference_time_s"].sum()),
        "total_pause_count": total_pause_count,
        "total_pause_duration_s": total_pause_duration_s,
        "mean_pause_duration_s": mean_pause_duration_s,
    }

    for column in [
        "mean_rms",
        "std_rms",
        "speech_activity_ratio",
        "embedding_std",
    ]:
        summary[f"{column}_mean"] = (
            float(included[column].mean()) if not included.empty else 0.0
        )
        summary[f"{column}_std"] = (
            float(included[column].std(ddof=0)) if not included.empty else 0.0
        )

    return pd.DataFrame([summary])


def save_sample_outputs(
    df: pd.DataFrame, summary: pd.DataFrame, output_dir: str | Path, sample_id: str
) -> None:
    sample_dir = Path(output_dir) / sample_id
    sample_dir.mkdir(parents=True, exist_ok=True)
    df.to_csv(sample_dir / "chunk_indicators.csv", index=False)
    summary.to_csv(sample_dir / "session_summary.csv", index=False)


def save_combined_outputs(
    all_chunks: pd.DataFrame, all_summaries: pd.DataFrame, output_dir: str | Path
) -> None:
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    all_chunks.to_csv(output_dir / "all_chunk_indicators.csv", index=False)
    all_summaries.to_csv(output_dir / "all_session_summaries.csv", index=False)
    print(f"Saved combined CSV outputs to {output_dir}")

In [ ]:
def plot_chunk_timeline(df: pd.DataFrame) -> None:
    fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
    axes[0].plot(df["start_time_s"], df["mean_rms"], marker="o")
    axes[0].set_ylabel("Mean RMS")
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(
        df["start_time_s"], df["speech_activity_ratio"], marker="o", color="tab:green"
    )
    axes[1].set_xlabel("Start time (s)")
    axes[1].set_ylabel("Speech activity ratio")
    axes[1].set_ylim(0, 1)
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

## 7. Batch Run Evaluation

In [ ]:
def copy_file(row):
    src = Path(str(row["wav_drive_path"]))
    if not src.exists():
        return None, f"Drive WAV path missing for {row['sample_id']}: {src}"

    dst = LOCAL_EVAL_WAV_DIR / src.name
    shutil.copy2(src, dst)

    return dst, None

In [ ]:
from google.colab import drive
from concurrent.futures import ThreadPoolExecutor

drive.mount("/content/drive")

if not MANIFEST_PATH.exists():
    raise FileNotFoundError(
        f"Dataset manifest not found: {MANIFEST_PATH}. "
        "Run preprocessing_pipeline_random_samples.ipynb first, or point MANIFEST_PATH at the Drive manifest copy."
    )
if not MODEL_PATH.exists():
    raise FileNotFoundError(f"Model file not found: {MODEL_PATH}")

manifest = pd.read_csv(MANIFEST_PATH)
manifest = manifest[manifest["status"].astype(str).str.startswith("completed")].copy()
manifest = manifest[
    manifest["wav_drive_path"].fillna("").astype(str).str.len() > 0
].copy()

if manifest.empty:
    raise RuntimeError(
        "No completed manifest rows with wav_drive_path were found. "
        "The eval pipeline intentionally uses Drive WAVs as the source of truth."
    )

records = manifest.to_dict("records")
print(f"Pre-fetching {len(records)} files from Drive to local disk.")

local_paths = {}

with ThreadPoolExecutor(max_workers=12) as executor:
    futures = {executor.submit(copy_file, r): r["sample_id"] for r in records}
    for future in futures:
        sid = futures[future]
        dst, err = future.result()
        if err:
            raise FileNotFoundError(err)
        local_paths[sid] = dst

print("All files cached locally, starting inference.")

interpreter = load_interpreter(MODEL_PATH)
chunk_frames: list[pd.DataFrame] = []
summary_frames: list[pd.DataFrame] = []

for row in records:
    sample_id = str(row["sample_id"])
    local_wav_path = local_paths[sample_id]

    print(f"Evaluating {sample_id}.")
    df, total_duration_s = evaluate_audio_file(
        local_wav_path, MODEL_PATH, interpreter=interpreter
    )
    summary = session_summary(df, total_duration_s)

    metadata = {
        "sample_id": sample_id,
        "source_filename": row.get("source_filename", ""),
        "wav_drive_path": str(row["wav_drive_path"]),
        "wav_local_work_path": str(local_wav_path),
        "window_start_s": row.get("window_start_s", ""),
        "window_end_s": row.get("window_end_s", ""),
        "window_duration_s": row.get("window_duration_s", ""),
    }

    for key, value in metadata.items():
        df[key] = value
        summary[key] = value

    save_sample_outputs(df, summary, OUTPUT_DIR, sample_id)
    chunk_frames.append(df)
    summary_frames.append(summary)

all_chunks = pd.concat(chunk_frames, ignore_index=True)
all_summaries = pd.concat(summary_frames, ignore_index=True)
save_combined_outputs(all_chunks, all_summaries, OUTPUT_DIR)

display(all_summaries)
print(f"Evaluated {len(summary_frames)} samples.")